# Phase 1 Pilot 分析

這個 notebook 用來互動式檢視 Phase 1 管線的產出：

1. `snapshot_metadata.csv` — 快照下載狀況
2. `static_resources.csv` — 解析出的資源清單
3. `tdi_scores.csv` — TDI 分數

**注意**：請先跑完管線（`python -m scripts.phase1_run_pipeline --sample dataset/processed/samples/pilot_30.csv`），再開這個 notebook。

在 VS Code 開啟時，請選擇專案的 `.venv` 作為 kernel。

In [ ]:
# 讓 notebook 找得到專案模組（notebook 在 analysis/ 底下，要把根目錄加進 sys.path）
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from config.paths import SNAPSHOT_METADATA_CSV, STATIC_RESOURCES_CSV, TDI_SCORES_CSV

pd.set_option("display.max_columns", None)

## 1. 快照下載狀況

In [ ]:
meta = pd.read_csv(SNAPSHOT_METADATA_CSV)
print(f"任務數：{len(meta)}，成功：{(meta['downloaded'] == 1).sum()}")
print("\n各年份成功率：")
display(meta.groupby("target_year")["downloaded"].agg(["sum", "count", "mean"]))
print("\n失敗原因：")
display(meta.loc[meta["downloaded"] != 1, "error_message"].value_counts())

## 2. 資源清單概覽

In [ ]:
res = pd.read_csv(STATIC_RESOURCES_CSV)
print(f"總資源數：{len(res)}，第三方：{res['is_third_party'].sum()}")
print("\n各類型資源數：")
display(res.groupby(["resource_type", "is_third_party"]).size().unstack(fill_value=0))
print("\n最常見的第三方網域 Top 20：")
display(res[res["is_third_party"] == 1]["registrable_domain"].value_counts().head(20))

## 3. TDI 分數

In [ ]:
tdi = pd.read_csv(TDI_SCORES_CSV)
print("各年份 TDI 描述統計：")
display(tdi.groupby("year")["tdi"].describe())
print("\nTDI 最高的 10 個 site-year：")
display(tdi.sort_values("tdi", ascending=False).head(10))